# 아산페이 매출 분석 - 뛰어야산다2 방송효과
> 2026년 1~2월 아산페이 데이터로 뛰어야산다2 (2026-01-12 방영) 효과 측정

**데이터:** 아산페이 1월 + 2월 (모바일 결제 + 가맹점 + 연령별)

**분석:**
1. 가맹점 → 읍면동/업종 매핑
2. 방송 노출 관광지 주변 매출 변화
3. 연령대별 소비 변화 (타겟 데모)
4. 전후 비교 + 시각화

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rc
import zipfile, os, re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    rc('font', family=font_manager.FontProperties(fname='C:/Windows/Fonts/malgun.ttf').get_name())
except:
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

COLORS = ['#2196F3','#FF5722','#4CAF50','#FFC107','#9C27B0','#00BCD4','#E91E63']
print('Ready')

In [ ]:
# ============================
# 경로 설정
# ============================
DATA_DIR = Path(r'C:\Users\HP\Desktop\01.데이터')
OUTPUT_DIR = Path(r'C:\Users\HP\Desktop\02.분석결과')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 아산페이 ZIP 파일 경로
ASANPAY_JAN = DATA_DIR / '아산페이(2026년 1월).zip'  # 1월
ASANPAY_FEB = DATA_DIR / '아산페이(26.2.).zip'       # 2월

# ZIP 파일명이 다를 수 있으니 자동 탐색
asanpay_zips = list(DATA_DIR.glob('아산페이*.zip'))
print(f'아산페이 ZIP: {len(asanpay_zips)}개')
for z in asanpay_zips:
    print(f'  {z.name} ({z.stat().st_size/1024:.0f}KB)')

# 뛰어야산다2 방송 정보
BROADCAST_DATE = pd.Timestamp('2026-01-12')
BROADCAST_NAME = '뛰어야산다2'
BROADCAST_SPOTS = ['신정호', '곡교천', '현충사', '온천', '캠핑장']
BROADCAST_DONGS = ['온양5동', '염치읍', '온양1동']  # 노출 관광지 인근 읍면동

In [ ]:
# ============================
# ZIP 해제 함수 (한글 파일명 처리)
# ============================
def extract_asanpay_zip(zip_path):
    """아산페이 ZIP에서 엑셀 파일 추출 (한글 파일명 cp949 처리)"""
    dfs = {}
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            # 한글 파일명 디코딩
            try:
                name = info.filename.encode('cp437').decode('cp949')
            except:
                name = info.filename
            
            if not name.endswith('.xlsx'):
                continue
            
            data = zf.read(info.filename)
            # 임시 파일로 저장 후 읽기
            import tempfile
            with tempfile.NamedTemporaryFile(suffix='.xlsx', delete=False) as tmp:
                tmp.write(data)
                tmp_path = tmp.name
            
            xl = pd.ExcelFile(tmp_path)
            for sheet in xl.sheet_names:
                key = f"{name}|{sheet}"
                dfs[key] = pd.read_excel(tmp_path, sheet_name=sheet)
                print(f"  {name} > {sheet}: {dfs[key].shape}")
            os.unlink(tmp_path)
    return dfs

print('함수 정의 완료')

---
## 1. 데이터 로드

In [ ]:
# 1월 + 2월 데이터 로드
all_dfs = {}
for zpath in asanpay_zips:
    print(f'\n=== {zpath.name} ===')
    dfs = extract_asanpay_zip(zpath)
    # 월 태깅
    month = '01' if '1월' in zpath.name else '02' if '2' in zpath.name else 'unknown'
    for key, df in dfs.items():
        all_dfs[f"{month}|{key}"] = df

print(f'\n전체 시트: {len(all_dfs)}개')
for k, v in all_dfs.items():
    print(f'  {k}: {v.shape}')

In [ ]:
# 시트 분류 (결제/가맹점/구매/지류)
payment_dfs = {}  # 모바일 결제
purchase_dfs = {} # 모바일 구매 (연령대별)
merchant_dfs = {} # 가맹점 마스터
paper_dfs = {}    # 지류

for key, df in all_dfs.items():
    month = key.split('|')[0]
    cols_str = ' '.join(str(c) for c in df.columns)
    
    if '가맹점ID' in cols_str and '총금액' in cols_str:
        payment_dfs[month] = df
        print(f'결제: {key} ({len(df):,}행)')
    elif '연령대' in cols_str and '구매금액' in cols_str:
        purchase_dfs[month] = df
        print(f'구매: {key} ({len(df):,}행)')
    elif '가맹점ID' in cols_str and '행정동' in cols_str:
        merchant_dfs[month] = df
        print(f'가맹점: {key} ({len(df):,}행)')
    elif '환전' in cols_str or '환전금액' in key:
        paper_dfs[f"{month}_{key}"] = df
        print(f'지류: {key} ({len(df):,}행)')
    elif '연령' in cols_str and '판매금액' in cols_str:
        purchase_dfs[f"{month}_paper"] = df
        print(f'지류구매: {key} ({len(df):,}행)')
    else:
        print(f'미분류: {key} | 컬럼: {list(df.columns)[:5]}')

---
## 2. 가맹점 → 읍면동/업종 매핑

In [ ]:
# 가맹점 마스터 통합 (1월 + 2월)
merchants = []
for month, df in merchant_dfs.items():
    df = df.copy()
    df['month'] = month
    merchants.append(df)

if merchants:
    df_merchant = pd.concat(merchants, ignore_index=True)
    # 중복 제거 (같은 가맹점ID)
    df_merchant = df_merchant.drop_duplicates(subset='가맹점ID', keep='last')
    print(f'가맹점 마스터: {len(df_merchant):,}개')
    
    # 행정동 정리
    df_merchant['행정동_clean'] = df_merchant['행정동'].replace('[NULL]', np.nan)
    # 주소에서 읍면동 추출 (행정동이 NULL인 경우)
    def extract_dong(row):
        if pd.notna(row['행정동_clean']) and row['행정동_clean'] != '[NULL]':
            return row['행정동_clean']
        addr = str(row.get('기본주소', ''))
        # "아산시 OO읍/면/동" 패턴
        m = re.search(r'아산시\s+(\S+[읍면동])', addr)
        if m:
            return m.group(1)
        return None
    
    df_merchant['dong'] = df_merchant.apply(extract_dong, axis=1)
    
    print(f'\n읍면동 분포:')
    display(df_merchant['dong'].value_counts().head(15))
    print(f'\n업종 분포:')
    display(df_merchant['분류'].value_counts().head(15))
else:
    df_merchant = pd.DataFrame()
    print('가맹점 데이터 없음')

In [ ]:
# 결제 데이터 + 가맹점 조인
payments = []
for month, df in payment_dfs.items():
    df = df.copy()
    df['month'] = month
    payments.append(df)

if payments and len(df_merchant) > 0:
    df_payment = pd.concat(payments, ignore_index=True)
    print(f'결제 데이터: {len(df_payment):,}행')
    
    # 가맹점 정보 조인
    merchant_lookup = df_merchant[['가맹점ID', 'dong', '분류']].drop_duplicates(subset='가맹점ID')
    df_pay = df_payment.merge(merchant_lookup, on='가맹점ID', how='left')
    
    # 금액 정리 (NaN → 0)
    for col in ['QR결제금액', 'QR결제건수', '카드결제금액', '카드결제건수', '총금액', '총결제건수']:
        if col in df_pay.columns:
            df_pay[col] = pd.to_numeric(df_pay[col], errors='coerce').fillna(0)
    
    print(f'조인 후: {len(df_pay):,}행')
    print(f'읍면동 매핑률: {df_pay["dong"].notna().mean():.1%}')
    display(df_pay.head())
else:
    df_pay = pd.DataFrame()
    print('결제 또는 가맹점 데이터 없음')

---
## 3. 읍면동별 매출 비교 (1월 vs 2월)

In [ ]:
# 읍면동별 월별 매출 집계
if len(df_pay) > 0:
    dong_monthly = df_pay.groupby(['month', 'dong']).agg(
        total_amt=('총금액', 'sum'),
        total_cnt=('총결제건수', 'sum'),
        merchant_cnt=('가맹점ID', 'nunique'),
    ).reset_index()
    
    # 피벗: 1월 vs 2월
    pivot = dong_monthly.pivot_table(
        values=['total_amt', 'total_cnt'],
        index='dong', columns='month', fill_value=0
    )
    pivot.columns = [f'{col[0]}_{col[1]}' for col in pivot.columns]
    
    # 변화율
    if 'total_amt_01' in pivot.columns and 'total_amt_02' in pivot.columns:
        pivot['amt_change_pct'] = ((pivot['total_amt_02'] - pivot['total_amt_01']) 
                                    / pivot['total_amt_01'] * 100).round(1)
        pivot['cnt_change_pct'] = ((pivot['total_cnt_02'] - pivot['total_cnt_01']) 
                                    / pivot['total_cnt_01'] * 100).round(1)
    
    pivot = pivot.sort_values('total_amt_02' if 'total_amt_02' in pivot.columns else pivot.columns[0], 
                               ascending=False)
    
    # 방송 노출 읍면동 표시
    pivot['방송노출'] = pivot.index.isin(BROADCAST_DONGS).map({True: 'O', False: ''})
    
    pivot.to_csv(OUTPUT_DIR / 'asanpay_dong_monthly.csv', encoding='utf-8-sig')
    print('읍면동별 1월 vs 2월 매출:')
    display(pivot)
else:
    print('데이터 없음')

In [ ]:
# 시각화: 방송 노출 vs 미노출 읍면동 비교
if len(df_pay) > 0:
    df_pay['is_broadcast_area'] = df_pay['dong'].isin(BROADCAST_DONGS)
    
    group = df_pay.groupby(['month', 'is_broadcast_area']).agg(
        total_amt=('총금액', 'sum'),
        total_cnt=('총결제건수', 'sum'),
    ).reset_index()
    group['label'] = group['is_broadcast_area'].map({True: '방송노출 지역', False: '기타 지역'})
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for idx, (metric, title) in enumerate([('total_amt', '총 매출액'), ('total_cnt', '총 결제건수')]):
        ax = axes[idx]
        pv = group.pivot(index='label', columns='month', values=metric).fillna(0)
        pv.plot(kind='bar', ax=ax, color=['#2196F3', '#FF5722'])
        ax.set_title(f'{title} (1월 vs 2월)', fontweight='bold')
        ax.set_xlabel('')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        ax.legend(['1월', '2월'])
        ax.grid(True, alpha=0.3, axis='y')
        # 값 표시
        for p in ax.patches:
            ax.annotate(f'{p.get_height():,.0f}', (p.get_x()+p.get_width()/2., p.get_height()),
                        ha='center', va='bottom', fontsize=8)
    
    plt.suptitle(f'아산페이: {BROADCAST_NAME} 방송 노출 지역 vs 기타\n(방영일: {BROADCAST_DATE.strftime("%Y-%m-%d")})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_asanpay_broadcast_area.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 4. 업종별 매출 변화

In [ ]:
# 관광 관련 업종 필터
TOURISM_BIZ = ['음식점', '숙박', '관광', '레저', '카페', '편의점/슈퍼/마트', '주유소']

if len(df_pay) > 0:
    biz_monthly = df_pay.groupby(['month', '분류']).agg(
        total_amt=('총금액', 'sum'),
        total_cnt=('총결제건수', 'sum'),
    ).reset_index()
    
    biz_pivot = biz_monthly.pivot_table(
        values='total_amt', index='분류', columns='month', fill_value=0
    )
    if '01' in biz_pivot.columns and '02' in biz_pivot.columns:
        biz_pivot['변화율(%)'] = ((biz_pivot['02'] - biz_pivot['01']) / biz_pivot['01'] * 100).round(1)
        biz_pivot = biz_pivot.sort_values('02', ascending=False)
    
    biz_pivot.to_csv(OUTPUT_DIR / 'asanpay_biz_monthly.csv', encoding='utf-8-sig')
    print('업종별 1월 vs 2월:')
    display(biz_pivot.head(20))
    
    # 관광 업종만 시각화
    tourism = biz_pivot[biz_pivot.index.str.contains('|'.join(TOURISM_BIZ), na=False)]
    if len(tourism) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        x = range(len(tourism))
        w = 0.35
        if '01' in tourism.columns:
            ax.bar([i-w/2 for i in x], tourism['01']/1e6, w, label='1월', color='#2196F3')
        if '02' in tourism.columns:
            ax.bar([i+w/2 for i in x], tourism['02']/1e6, w, label='2월', color='#FF5722')
        ax.set_xticks(list(x))
        ax.set_xticklabels(tourism.index, rotation=45, ha='right')
        ax.set_title('관광 관련 업종 아산페이 매출 (1월 vs 2월)', fontweight='bold')
        ax.set_ylabel('매출 (백만원)')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'fig_asanpay_tourism_biz.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# 방송 노출 지역의 업종별 변화 (처치군 세분화)
if len(df_pay) > 0:
    treat = df_pay[df_pay['is_broadcast_area']]
    treat_biz = treat.groupby(['month', '분류']).agg(
        total_amt=('총금액', 'sum'), total_cnt=('총결제건수', 'sum')
    ).reset_index()
    
    treat_pivot = treat_biz.pivot_table(
        values='total_amt', index='분류', columns='month', fill_value=0
    )
    if '01' in treat_pivot.columns and '02' in treat_pivot.columns:
        treat_pivot['변화율(%)'] = ((treat_pivot['02'] - treat_pivot['01']) / treat_pivot['01'] * 100).round(1)
        treat_pivot = treat_pivot.sort_values('02', ascending=False)
    
    print(f'방송 노출 지역({\', \'.join(BROADCAST_DONGS)}) 업종별 변화:')
    display(treat_pivot.head(15))

---
## 5. 연령대별 소비 변화

In [ ]:
# 연령대별 구매 데이터 (뛰어야산다2 타겟: 2030)
purchases = []
for key, df in purchase_dfs.items():
    month = key.split('_')[0] if '_' in key else key
    df = df.copy()
    df['month'] = month
    df['source'] = 'paper' if 'paper' in key else 'mobile'
    purchases.append(df)

if purchases:
    # 모바일 구매만 (컬럼: 연령대, 구매금액, 구매할인액, 총구매금액, 구매건수)
    mobile_purchases = [df for df in purchases if df['source'].iloc[0] == 'mobile']
    if mobile_purchases:
        df_purchase = pd.concat(mobile_purchases, ignore_index=True)
        
        # 연령대 컬럼 통일
        age_col = [c for c in df_purchase.columns if '연령' in str(c)][0]
        amt_col = [c for c in df_purchase.columns if '총구매금액' in str(c) or '구매금액' in str(c)][0]
        cnt_col = [c for c in df_purchase.columns if '구매건수' in str(c)][0]
        
        age_pivot = df_purchase.pivot_table(
            values=[amt_col, cnt_col], index=age_col, columns='month', aggfunc='sum', fill_value=0
        )
        
        print('연령대별 구매 (1월 vs 2월):')
        display(age_pivot)
        
        # 시각화
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 금액
        ax = axes[0]
        amt_data = df_purchase.groupby(['month', age_col])[amt_col].sum().reset_index()
        amt_pv = amt_data.pivot(index=age_col, columns='month', values=amt_col).fillna(0)
        amt_pv.plot(kind='bar', ax=ax, color=['#2196F3', '#FF5722'])
        ax.set_title('연령대별 구매금액', fontweight='bold')
        ax.set_ylabel('금액 (원)')
        ax.legend(['1월', '2월'])
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        ax.grid(True, alpha=0.3, axis='y')
        
        # 건수
        ax = axes[1]
        cnt_data = df_purchase.groupby(['month', age_col])[cnt_col].sum().reset_index()
        cnt_pv = cnt_data.pivot(index=age_col, columns='month', values=cnt_col).fillna(0)
        cnt_pv.plot(kind='bar', ax=ax, color=['#2196F3', '#FF5722'])
        ax.set_title('연령대별 구매건수', fontweight='bold')
        ax.set_ylabel('건수')
        ax.legend(['1월', '2월'])
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.suptitle(f'아산페이 연령대별 소비 변화\n(뛰어야산다2 타겟: 2030세대)',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'fig_asanpay_age.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        # 변화율 계산
        if '01' in amt_pv.columns and '02' in amt_pv.columns:
            amt_pv['변화율(%)'] = ((amt_pv['02'] - amt_pv['01']) / amt_pv['01'] * 100).round(1)
            print('\n연령대별 구매금액 변화율:')
            display(amt_pv)
else:
    print('구매 데이터 없음')

---
## 6. DID 분석 (방송노출 vs 미노출)

In [ ]:
# 간단 DID: (처치군 2월 - 처치군 1월) - (대조군 2월 - 대조군 1월)
if len(df_pay) > 0:
    did_data = df_pay.groupby(['month', 'is_broadcast_area']).agg(
        avg_amt_per_merchant=('총금액', 'mean'),
        total_amt=('총금액', 'sum'),
    ).reset_index()
    
    print('DID 데이터:')
    display(did_data)
    
    # DID 계산
    try:
        treat_01 = did_data[(did_data['month']=='01') & (did_data['is_broadcast_area']==True)]['avg_amt_per_merchant'].values[0]
        treat_02 = did_data[(did_data['month']=='02') & (did_data['is_broadcast_area']==True)]['avg_amt_per_merchant'].values[0]
        ctrl_01 = did_data[(did_data['month']=='01') & (did_data['is_broadcast_area']==False)]['avg_amt_per_merchant'].values[0]
        ctrl_02 = did_data[(did_data['month']=='02') & (did_data['is_broadcast_area']==False)]['avg_amt_per_merchant'].values[0]
        
        did_effect = (treat_02 - treat_01) - (ctrl_02 - ctrl_01)
        
        print(f'\n=== DID 결과 ===')
        print(f'처치군 (방송노출): 1월 {treat_01:,.0f} → 2월 {treat_02:,.0f} (차이: {treat_02-treat_01:+,.0f})')
        print(f'대조군 (미노출) : 1월 {ctrl_01:,.0f} → 2월 {ctrl_02:,.0f} (차이: {ctrl_02-ctrl_01:+,.0f})')
        print(f'\nDID 효과 (순수 방송효과): {did_effect:+,.0f}원/가맹점')
        
        if did_effect > 0:
            treat_merchants = df_pay[(df_pay['is_broadcast_area']) & (df_pay['month']=='02')]['가맹점ID'].nunique()
            total_effect = did_effect * treat_merchants
            print(f'방송노출 지역 가맹점 수: {treat_merchants}개')
            print(f'총 순수 방송효과 추정: {total_effect:,.0f}원 ({total_effect/1e6:,.1f}백만원)')
    except (IndexError, KeyError) as e:
        print(f'DID 계산 실패: {e}')

---
## 7. 종합 요약

In [ ]:
print('=' * 60)
print(f'아산페이 분석 완료 - {BROADCAST_NAME} 방송효과')
print('=' * 60)
print(f'방영일: {BROADCAST_DATE.strftime("%Y-%m-%d")}')
print(f'노출 관광지: {", ".join(BROADCAST_SPOTS)}')
print(f'분석 대상 읍면동: {", ".join(BROADCAST_DONGS)}')
print(f'\n데이터: 아산페이 2026년 1월 + 2월')
print(f'비교: 1월(방송 전반) vs 2월(방송 후)')
print(f'\n참고: 1월 12일 방영이므로 1월 데이터에도 방송 후 효과가 일부 포함됨')
print(f'      더 정밀한 분석은 일별 데이터 필요')

print(f'\n생성된 파일:')
for f in sorted(OUTPUT_DIR.glob('asanpay*')):
    try:
        print(f'  {f.name} ({f.stat().st_size/1024:.0f}KB)')
    except: pass
for f in sorted(OUTPUT_DIR.glob('fig_asanpay*')):
    try:
        print(f'  {f.name} ({f.stat().st_size/1024:.0f}KB)')
    except: pass